# Aula 3 · Revisão — o caminho até aqui

Três aulas, três perguntas:

| Aula | Pergunta | Resposta |
|------|----------|----------|
| 1 | **O que o dado diz?** | Pedido atrasado vira review ruim |
| 2 | **Dá pra prever?** | Sim — um modelo acerta ~86% |
| 3 (hoje) | **Como isso vira um serviço?** | Banco + pipeline + dashboard |

Este notebook refaz o caminho em 10 minutos. Tudo que aparece aqui
volta hoje — com outra roupa.

## 1. O dado da Aula 1: o Olist

In [ ]:
import urllib.request, pathlib
import pandas as pd

URL = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula1-olist-2026-v1/olist.parquet"
PATH = pathlib.Path("olist.parquet")

if not PATH.exists():
    print("Baixando olist.parquet (~8 MB)...")
    urllib.request.urlretrieve(URL, PATH)
    print(f"✅ Baixado: {PATH.stat().st_size / 1e6:.1f} MB")
else:
    print(f"Já está baixado: {PATH.stat().st_size / 1e6:.1f} MB")

df = pd.read_parquet(PATH)
print(f"{len(df):,} pedidos, {df.shape[1]} colunas")
df[["order_id", "customer_state", "product_category_en",
    "price", "delivery_days", "delivered_late", "review_score"]].head()

Um Parquet com ~96 mil pedidos reais de e-commerce. Na Aula 1 a gente
aprendeu a **perguntar** a esse arquivo — primeiro com pandas, depois com SQL.

## 2. A descoberta que guia o curso

A pergunta da Aula 1: **entregar atrasado muda a nota que o cliente dá?**

A resposta veio por SQL (DuckDB, direto no arquivo):

In [ ]:
import duckdb

duckdb.sql("""
    SELECT delivered_late,
           round(avg(review_score), 2) AS nota_media,
           count(*)                    AS pedidos
    FROM 'olist.parquet'
    WHERE delivered_late IS NOT NULL
    GROUP BY delivered_late
""").df()

In [ ]:
notas = (df.dropna(subset=["delivered_late"])
           .groupby("delivered_late")["review_score"].mean())
ax = notas.plot(kind="bar", color=["#10834A", "#E03131"], figsize=(7, 3.5),
                edgecolor="white", rot=0)
ax.set_title("Atrasou? A nota despenca.", fontweight="bold")
ax.set_xticklabels(["entregou no prazo", "atrasou"])
ax.set_ylabel("nota média (1–5)")
for i, v in enumerate(notas):
    ax.text(i, v + 0.08, f"{v:.2f}", ha="center", fontweight="bold")

**A descoberta:** o atraso derruba a nota de **~4,3 para ~2,3**. Review ruim
espanta cliente novo — então atraso custa dinheiro **duas vezes**.

Guarde esta query: ela reaparece **hoje**, idêntica, mudando só uma linha —
o `FROM` deixa de ser um arquivo e vira uma tabela no Postgres.

## 3. Aula 2: de olhar o passado a prever o futuro

Saber que atrasou → nota ruim é olhar pelo **retrovisor**. A Aula 2 perguntou:
dá pra saber **antes** da nota chegar?

- Transformamos texto em número (**TF-IDF**) e juntamos com preço, frete, prazo…
- Treinamos um modelo (regressão logística) com dado **rotulado**
- Ele acerta **~86%** — e virou uma **API**: manda um pedido, volta uma previsão

O resultado do modelo aplicado ao Olist inteiro está num Parquet novo —
**o dado de hoje**, com uma coluna a mais:

In [ ]:
URL3 = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula3-olist-risco-2026-v1/olist_com_risco.parquet"
PATH3 = pathlib.Path("olist_com_risco.parquet")

if not PATH3.exists():
    print("Baixando olist_com_risco.parquet (~4 MB)...")
    urllib.request.urlretrieve(URL3, PATH3)
    print(f"✅ Baixado: {PATH3.stat().st_size / 1e6:.1f} MB")
else:
    print(f"Já está baixado: {PATH3.stat().st_size / 1e6:.1f} MB")

risco = pd.read_parquet(PATH3)
print(f"{len(risco):,} pedidos | {risco['risco_review'].sum():,} em risco "
      f"({risco['risco_review'].mean():.1%})")
risco.head()

**O modelo funciona?** Compare a previsão com a nota que o cliente deu de verdade:

In [ ]:
nota_real = (risco.dropna(subset=["review_score"])
             .groupby("risco_review")["review_score"].mean())
ax = nota_real.plot(kind="bar", color=["#10834A", "#E03131"], figsize=(7, 3.5),
                    edgecolor="white", rot=0)
ax.set_title("O que o modelo chamou de 'risco' virou nota baixa mesmo?",
             fontweight="bold")
ax.set_xticklabels(["sem risco", "em risco"])
ax.set_ylabel("nota média REAL (1–5)")
for i, v in enumerate(nota_real):
    ax.text(i, v + 0.08, f"{v:.2f}", ha="center", fontweight="bold")

Os pedidos que o modelo marcou como risco tiveram, de fato, nota média
muito pior. **A previsão funciona** — e hoje ela vira uma coluna da tabela
(`risco_review`) que alimenta o dashboard.

## 4. E o robô da Aula 2: o n8n

Também montamos um **workflow**: um robô que chama uma API no horário agendado
e coleta o que ela devolver — sem ninguém apertar nada.

Hoje a API devolve pedidos neste formato:

```json
{ "id": "novo-a1b2c3d4e5f6", "customer_state": "SP",
  "category": "housewares", "price": 89.9, "delivery_days": 9 }
```

Na Aula 2 o robô coletava e… **mostrava o JSON na tela**. E aí?

## 5. A pergunta que abre a aula de hoje

> O dado existe (Aula 1). A previsão existe (Aula 2). O robô coleta sozinho.
>
> **Mas onde esse dado ATERRISSA?**

Num arquivo? E se o robô grava às 8h enquanto alguém lê? E se três pessoas
abrem ao mesmo tempo? E quem NÃO sabe Python — como vê isso tudo?

**Hoje:** o dado aterrissa num **banco de dados** (Postgres), o robô grava nele
sozinho (n8n) e um **dashboard** (Metabase) mostra para quem decide. Bora.